In [1]:
import pandas as pd
excel_file = 'project/dataset/SeriesReport-20251222061308_06c630.xlsx'
csv_file = 'project/dataset/SeriesReport-20251222061308_06c630.csv'
df = pd.read_excel(excel_file)
df.to_csv(csv_file, index=False, encoding='utf-8')
print(f"文件已转换: {csv_file}")

文件已转换: project/dataset/SeriesReport-20251222061308_06c630.csv


c:\Users\lzl\miniconda3\envs\dev\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [ ]:
df_industry = pd.read_csv('project/dataset/Unemployment_industry.csv')
series_name = pd.read_csv('project/dataset/series_name.csv')

merged_df = df_industry.merge(series_name, left_on='Series ID', right_on='series_id', how='left')

# 删除可能存在的冗余列（存在则删除）
merged_df = merged_df.drop(columns=['series_title', 'labor_force_experience'], errors='ignore')

merged_csv = 'project/dataset/Unemployment_industry_merged.csv'
merged_df.to_csv(merged_csv, index=False, encoding='utf-8')
print(f"已合并并保存: {merged_csv}")
# ...existing code...
# print(merged_df['industry'].unique())

已合并并保存: project/dataset/Unemployment_industry_merged.csv


In [16]:
def process_alignment(file_a_path, file_b_path, output_path):
    """
    file_a_path: project/dataset/Unemployment_industry_merged.csv
    file_b_path: project/dataset/AIOE_DataAppendix__Appendix_B.csv
    """

    # 1. 加载数据
    df_a = pd.read_csv(file_a_path) # 假设列名为 'Industry'
    df_b = pd.read_csv(file_b_path)

    # 2. 定义 NAICS 到 BLS Supersector 的映射逻辑 (自动化分类)
    def map_naics_to_sector(naics):
        n2 = int(str(naics)[:2])
        n3 = int(str(naics)[:3])
        
        if n2 == 11: return 'Agriculture and related industries'
        if n2 == 21: return 'Mining, quarrying, and oil and gas extraction'
        if n2 == 23: return 'Construction'
        if n2 == 22 or n2 in [48, 49]: return 'Transportation and utilities'
        if n2 == 42 or n2 in [44, 45]: return 'Wholesale and retail trade'
        if n2 == 51: return 'Information'
        if n2 in [52, 53]: return 'Financial activities'
        if n2 in [54, 55, 56]: return 'Professional and business services'
        if n2 in [61, 62]: return 'Education and health services'
        if n2 in [71, 72]: return 'Leisure and hospitality'
        if n2 == 81: return 'Other services'
        if n2 in [31, 32, 33]:
            # 区分耐用与非耐用制造业 (基于 NAICS 3位代码)
            durable_codes = [321, 327, 331, 332, 333, 334, 335, 336, 337, 339]
            return 'Durable goods manufacturing' if n3 in durable_codes else 'Nondurable goods manufacturing'
        return 'Other'

    # 3. 在文件 B 中应用映射并计算均值
    df_b['Mapped_Sector'] = df_b['NAICS'].apply(map_naics_to_sector)
    
    # 计算每个行业的 AI 暴露度均值 (Aggregated Score)
    sector_scores = df_b.groupby('Mapped_Sector')['AIIE'].mean().reset_index()
    sector_scores.columns = ['industry', 'AI_Exposure_Score']

    # 4. 对齐并更新文件 A
    # 使用左连接（Left Join）确保文件 A 的原始顺序和结构不变
    updated_df_a = pd.merge(df_a, sector_scores, on='industry', how='left')
    
    # 5. 特殊处理：计算 Nonagriculture industries (非农行业) 的总体平均值
    non_agri_mask = updated_df_a['industry'] != 'Agriculture and related industries'
    overall_avg = updated_df_a.loc[non_agri_mask, 'AI_Exposure_Score'].mean()
    
    # 填补 'Nonagriculture industries' 行的值
    updated_df_a.loc[updated_df_a['industry'] == 'Nonagriculture industries', 'AI_Exposure_Score'] = overall_avg
    # 6. 保存结果
    updated_df_a.to_csv(output_path, index=False)
    print(f"对齐完成！更新后的数据已保存至: {output_path}")
    return updated_df_a

# 使用示例
process_alignment('project/dataset/Unemployment_industry_merged.csv', 'project/dataset/AIOE_DataAppendix__Appendix_B.csv', 'project/dataset/Updated_Unemployment_industry.csv')

对齐完成！更新后的数据已保存至: project/dataset/Updated_Unemployment_industry.csv


,Series ID,Jan 2022,Feb 2022,Mar 2022,Apr 2022,May 2022,Jun 2022,Jul 2022,Aug 2022,Sep 2022,...,Aug 2025,Sep 2025,Nov 2025,series_id,labor_force_status,type_of_data,age,class_of_worker,industry,AI_Exposure_Score
0,LNU03000000,7207.0,6782.0,6168.0,5458.0,5548.0,6334.0,6255.0,6256.0,5460.0,...,7747.0,7324.0,7401.0,LNU03000000,Unemployed,Number in thousands,16 years and over,NaN,NaN,NaN
1,LNU04000000,4.4,4.1,3.8,3.3,3.4,3.8,3.8,3.8,3.3,...,4.5,4.3,4.3,LNU04000000,Unemployment rate,Percent or rate,16 years and over,NaN,NaN,NaN
2,LNU03032229,5822.0,5479.0,5035.0,4288.0,4353.0,4607.0,4475.0,4786.0,4297.0,...,5610.0,5587.0,5521.0,LNU03032229,Unemployed,Number in thousands,16 years and over,Private wage and salary workers,Nonagriculture industries,0.076906
3,LNU04032229,4.5,4.2,3.9,3.3,3.3,3.5,3.4,3.6,3.3,...,4.1,4.1,4.1,LNU04032229,Unemployment rate,Percent or rate,16 years and over,Private wage and salary workers,Nonagriculture industries,0.076906
4,LNU03032230,46.0,29.0,14.0,19.0,24.0,10.0,6.0,16.0,16.0,...,20.0,33.0,50.0,LNU03032230,Unemployed,Number in thousands,16 years and over,Private wage and salary workers,"Mining, quarrying, and oil and gas extraction",-0.677795
5,LNU04032230,8.4,5.0,2.6,3.4,4.1,1.6,0.8,2.6,2.5,...,4.0,6.8,9.0,LNU04032230,Unemployment rate,Percent or rate,16 years and over,Private wage and salary workers,"Mining, quarrying, and oil and gas extraction",-0.677795
6,LNU03032231,709.0,677.0,598.0,464.0,392.0,385.0,359.0,401.0,346.0,...,347.0,404.0,431.0,LNU03032231,Unemployed,Number in thousands,16 years and over,Private wage and salary workers,Construction,-0.996961
7,LNU04032231,7.1,6.7,6.0,4.6,3.8,3.7,3.5,3.9,3.4,...,3.2,3.8,4.1,LNU04032231,Unemployment rate,Percent or rate,16 years and over,Private wage and salary workers,Construction,-0.996961
8,LNU03032232,549.0,497.0,485.0,489.0,422.0,465.0,491.0,515.0,416.0,...,555.0,571.0,507.0,LNU03032232,Unemployed,Number in thousands,16 years and over,Private wage and salary workers,Manufacturing,NaN
9,LNU04032232,3.6,3.2,3.1,3.2,2.8,3.0,3.2,3.3,2.8,...,3.8,3.7,3.3,LNU04032232,Unemployment rate,Percent or rate,16 years and over,Private wage and salary workers,Manufacturing,NaN


In [17]:
# 清洗重复列（按列名与按内容完全相同），并保存为新文件
input_path = 'project/dataset/Updated_Unemployment_industry.csv'
output_path = 'project/dataset/Updated_Unemployment_industry_cleaned.csv'

df = pd.read_csv(input_path)

# 规范列名空白以避免“看似不同”的重复
original_columns = df.columns.tolist()
df.columns = df.columns.str.strip()

# 1) 按列名去重（保留首次出现）
name_dupe_mask = df.columns.duplicated()
name_dupe_cols = df.columns[name_dupe_mask].tolist()
df_no_name_dupes = df.loc[:, ~name_dupe_mask]

# 2) 按内容去重（完全相同的列，仅保留一列）
before_content_cols = set(df_no_name_dupes.columns)
df_cleaned = df_no_name_dupes.T.drop_duplicates().T
content_removed_cols = sorted(list(before_content_cols - set(df_cleaned.columns)))

# 保存结果
df_cleaned.to_csv(output_path, index=False, encoding='utf-8')

# 输出清洗信息
print(f'读取: {input_path}')
print(f'初始列数: {len(original_columns)}')
if name_dupe_cols:
    print(f'按列名移除 {len(name_dupe_cols)} 列: {name_dupe_cols}')
else:
    print('按列名未发现重复列')
if content_removed_cols:
    print(f'按内容移除 {len(content_removed_cols)} 列: {content_removed_cols}')
else:
    print('按内容未发现完全重复列')
print(f'清洗后列数: {df_cleaned.shape[1]}')
print(f'已保存: {output_path}')

读取: project/dataset/Updated_Unemployment_industry.csv
初始列数: 57
按列名未发现重复列
按内容移除 1 列: ['series_id']
清洗后列数: 56
已保存: project/dataset/Updated_Unemployment_industry_cleaned.csv
